[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/02_%E4%BB%A3%E8%A1%A8%E5%80%A4.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-02. 代表値（平均・中央値・最頻値）

**できるようになること**
- 平均・中央値・最頻値を計算して使い分けられる
- 外れ値に強いのは中央値、と理由つきで説明できる
- グループ別に代表値を比べられる

**前提**：`01_Python基礎`　/　**所要**：約25分　/　**レベル**：統計検定4級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画

# 日本語が文字化けするときは次の1行を有効にしてください
# plt.rcParams['font.family'] = 'Hiragino Sans'  # Macの場合
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

## 1. 3つの代表値

| 代表値 | 意味 | 強い点 / 弱い点 |
|---|---|---|
| 平均値 (mean) | 合計 ÷ 個数 | 計算しやすい / 極端な値に弱い |
| 中央値 (median) | 真ん中の値 | 極端な値に強い |
| 最頻値 (mode) | 最も多い値 | 質的データにも使える |

In [ ]:
math = df['数学']                          # 数学の列を取り出す
print('平均値:', math.mean().round(2))     # 平均値
print('中央値:', math.median())            # 中央値（真ん中の値）
print('最頻値:', math.mode().tolist())     # 最頻値（最も多い値）

## 2. 平均は「外れ値」に弱い

お金持ちが1人いるだけで平均年収が跳ね上がる、という話を確かめます。

In [ ]:
incomes = [300, 320, 350, 310, 330, 9000]  # 万円。最後の1人だけ大金持ち
print('平均:', np.mean(incomes))    # 1601.7万 … 1人に引っ張られ実感と合わない
print('中央値:', np.median(incomes)) # 325万 … こちらが実感に近い

> **外れ値があるときは中央値**のほうが「ふつうの人」を表します。
ニュースで「年収の中央値」が使われるのはこのためです。

## 3. クラスごとに平均を比べる

`groupby` を使うと、グループ別の集計が一発でできます。

In [ ]:
df.groupby('クラス')['数学'].mean().round(1)   # クラスごとの数学の平均

In [ ]:
plt.figure(figsize=(6, 4))                        # グラフの大きさ
df.groupby('クラス')['数学'].mean().plot(kind='bar')  # クラス別平均を棒グラフに
plt.ylabel('数学の平均点')                        # y軸ラベル
plt.title('クラス別の平均点')                     # タイトル
plt.xticks(rotation=0)                            # x軸目盛りを水平に
plt.show()

## 4. 分布の形で代表値の並び順が決まる

右に裾が長い分布（高いほうに極端な値がある）では、平均が裾に引っ張られて
**最頻値 ≤ 中央値 ≤ 平均値** の順になります。左に裾が長いと逆順です。
ヒストグラムの形を見れば、3つの大小関係が予想できます。

In [ ]:
rng = np.random.default_rng(0)
skew = np.round(rng.lognormal(3, 0.5, 2000))   # 右に裾が長い架空データ
mo = pd.Series(skew).mode().iloc[0]            # 最頻値
me = np.median(skew)                           # 中央値
mn = round(skew.mean(), 1)                     # 平均値
print('最頻値:', mo, '／中央値:', me, '／平均値:', mn)
plt.figure(figsize=(7,4))
plt.hist(skew, bins=40, edgecolor='white')
for v,c,lab in [(mo,'C1','最頻値'),(me,'C2','中央値'),(mn,'C3','平均値')]:
    plt.axvline(v, color=c, linestyle='--', label=lab)
plt.legend(); plt.title('右に裾が長い分布：最頻値 ≤ 中央値 ≤ 平均値'); plt.show()

> **平均と中央値の差で偏りが読めます。** 平均 > 中央値なら右裾（高い外れ値あり）、平均 < 中央値なら左裾。全部の数値を見なくても分布の偏りの向きがわかります。

> **よくある間違い**：「平均＝真ん中」ではありません。分布が左右非対称だと、平均は裾（極端な値）に引っ張られ、中央値とズレます（年収・運賃などが典型）。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
x = [60, 75, 80, 75, 90, 75, 55]
# Q1: x の中央値を ans に
ans = None   # 例: np.median(x)
_check('Q1 中央値', ans, np.median(x))

In [ ]:
x = [60, 75, 80, 75, 90, 75, 55]
# Q2: x の平均を ans に
ans = None
_check('Q2 平均', ans, np.mean(x))

In [ ]:
x2 = [3, 3, 4, 4, 4, 5, 12]
# Q3: x2 の最頻値（最も多い値）を ans に
ans = None   # 例: pd.Series(x2).mode().iloc[0]
_check('Q3 最頻値', ans, float(pd.Series(x2).mode().iloc[0]))

---
## 練習問題

**問1.** 次の7人のテスト点数の 平均・中央値・最頻値 を求めよ：
`[60, 75, 80, 75, 90, 75, 55]`

**問2.** `英語` について、クラスごとの中央値を求めよう。

**問3.** ある店の1日の来客数 `[20, 22, 21, 19, 200]` がある。
平均と中央値を計算し、「どちらがこの店のふつうの日を表すか」を説明しよう。

**問4.** ある植物の鉢ごとの発芽数 `[2,3,3,4,4,4,5,8,11,14,19,27]` について、最頻値・中央値・平均を求めよう。さらに **平均と中央値の大小** から、この分布が左右どちらに裾を引いているか答えよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[02_代表値 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/02_%E4%BB%A3%E8%A1%A8%E5%80%A4.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 | 強い/弱い |
|---|---|---|
| 平均値 | 合計÷個数 | 計算しやすい/外れ値に弱い |
| 中央値 | 真ん中の値 | 外れ値に強い |
| 最頻値 | 最も多い値 | 質的データにも使える |

In [ ]:
# チートシート（代表値）
s = df['数学']
s.mean(); s.median(); s.mode()        # 平均・中央値・最頻値
df.groupby('クラス')['数学'].mean()    # グループ別の平均

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "02_統計_4級/02_代表値"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")